# Scalar nonlocal AMB M00–M05 · ShellForce CNEEP

기존 `Corr_AMB2D_ShellForce.ipynb`의 **시뮬레이션 → 학습 → checkpoint → EPR/map/shell 비교** 흐름에 명세의 여섯 모델을 연결합니다. 원본 네트워크 파일은 수정하지 않습니다.

기본 물리 진단은 CPU의 **짧은 실행 점검(smoke)** 입니다. 학습을 켜면 LABPM 규모의 별도 독립 ensemble을 생성하며, 짧은 진단 trajectory를 학습에 재사용하지 않습니다. `RUN_EXTENDED`, `DO_TRAIN` 설정으로 긴 물리 실험과 NN 학습을 선택합니다. 짧은 실행의 유한한 출력이나 loss 감소는 stationarity, 유의한 EPR, kernel recovery의 검증이 아닙니다. 외부 설치·네트워크 요청은 실행하지 않습니다.

- 상태는 보존 scalar `phi` 하나, `h=M=1`; 시뮬레이션과 EPR 누적은 float64입니다.
- 물리 kernel의 전체 periodic support를 유지하고 NN 반경 밖은 `tail`로 별도 보고합니다.
- **물리 work / NN total / raw branch / known-model force projection**을 구분합니다. 음의 shell 기여도 보존합니다.
- 원본 문서: [COMMON_PROTOCOL](../data/amb_kernel/specs/COMMON_PROTOCOL.md), [모델 목록](../data/amb_kernel/specs/README.md), [실행 안내](../data/amb_kernel/README.md).

In [ ]:
from pathlib import Path
from dataclasses import asdict, replace
from datetime import datetime, timezone, timedelta
import json
import sys
from time import perf_counter

# Repository root, repository parent, and notebooks/ are all supported.
HERE = Path.cwd().resolve()
candidates = [HERE, *HERE.parents, HERE / 'CNEEP_v2']
ROOT = next((p for p in candidates if (p / 'models/NEEP_ShellForce_2D.py').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Run Jupyter inside CNEEP_v2 or its parent, or set ROOT explicitly.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from IPython.display import display
from tqdm.auto import tqdm
from data.amb_kernel.model import AMBKernelModel, MODEL_NAMES, pilot_config, control_configs, laplacian
from data.amb_kernel.experiment import RunConfig, run_experiment, load_experiment
from data.amb_kernel.dataset import (
    DatasetConfig, generate_dataset, load_dataset, resolve_simulation_device,
)
from data.amb_kernel.learning import (
    ArchitectureConfig, TrainingConfig,
    train_shell_force, load_checkpoint, evaluate_shell_force, projected_curl_check,
    source_provenance,
)

NOTEBOOK_STARTED = perf_counter()
STAGE_TIMINGS = []
SHOW_PROGRESS = True
BAR_FORMAT = '{desc}: {percentage:3.0f}%|{bar}| {n_fmt}/{total_fmt} [{elapsed} elapsed, ETA {remaining}, {rate_fmt}{postfix}]'

def record_stage(name, started):
    elapsed = perf_counter() - started
    STAGE_TIMINGS.append({'stage': name, 'elapsed_seconds': elapsed})
    tqdm.write(f'{name}: elapsed {timedelta(seconds=round(elapsed))} ({elapsed:.1f} s)')
    return elapsed

torch.set_num_threads(min(4, torch.get_num_threads()))
print('Repository:', ROOT)
print('Torch:', torch.__version__, '| quick diagnostics CPU/float64; dataset simulation device below')

## 1. 한 곳에서 바꾸는 실험 설정

기존 notebook의 `H=64`, `n_hidden=3`, `R=3`, ELU를 여섯 모델 모두에 고정합니다. 외부 shell은 `learned_absolute`, center mixing 없음, exclusive Chebyshev mask입니다. `n_components=1`, `ep_component_indices=(0,)`로 추가 관측 채널을 만들지 않습니다.

첨부 명세의 기준 `shell_force.py` SHA256과 현재 저장소의 파일은 다릅니다. 아래에서 실제 hash와 생성자 계약을 기록합니다. 현재 구현의 scalar 입력은 `[B,2,Ny,Nx]`이므로 저장된 `[T,1,Ny,Nx]`에서 **singleton 축만** 제거합니다. 같은 소스라고 주장하지 않으며 네트워크 수정 없이 기존 옵션을 사용합니다.

In [ ]:
MODEL_IDS = ('M00', 'M01', 'M02', 'M03', 'M04', 'M05')
ARCHITECTURE = ArchitectureConfig(n_channel=64, n_hidden=3, max_distance=3,
                                  shell_force_activation='elu')
NN_METADATA = {**vars(ARCHITECTURE.options()),
               'constructor_configuration': 'fixed existing Corr_AMB2D_ShellForce options'}
QUICK_N = 12
QUICK_RUN = RunConfig(seed=73031, dt=1e-3, burn_in_steps=64,
                      measurement_steps=128, observation_stride=1,
                      observation_lags=(1, 4, 16), block_steps=32,
                      max_distance=ARCHITECTURE.max_distance)
# Override physical parameters here, separately for each model; defaults follow the specs.
MODEL_OVERRIDES = {model_id: {} for model_id in MODEL_IDS}
# Example: MODEL_OVERRIDES['M01'] = dict(g2=0.2, phi_bar=0.3)

stamp = datetime.now(timezone(timedelta(hours=9))).strftime('%Y%m%d-%H%M%S-%f')
OUTPUT = ROOT / 'results' / 'amb_kernel' / stamp
OUTPUT.mkdir(parents=True, exist_ok=False)
print('Output:', OUTPUT)
display(pd.DataFrame([{'model_id': key, 'description': MODEL_NAMES[key]} for key in MODEL_IDS]))
display(source_provenance())
display(vars(ARCHITECTURE.options()))
(OUTPUT / 'notebook_settings.json').write_text(json.dumps({
    'purpose': 'short CPU smoke; physical validation not established',
    'architecture': vars(ARCHITECTURE.options()),
    'quick_run': asdict(QUICK_RUN), 'physical_overrides': MODEL_OVERRIDES,
    'source_provenance': source_provenance(),
}, indent=2), encoding='utf8')

## 2. 모든 모델의 simulator 실행과 저장

매 integration step에서 fixed-noise implicit midpoint를 풀고 work와 drift-minus-trace를 누적합니다. solver 실패를 clipping, noise 재추출, 자동 timestep 변경으로 숨기지 않습니다. 실패하면 원인을 확인하고 해당 run을 더 작은 `dt`로 처음부터 다시 실행합니다.

각 run에는 `manifest.json`, `kernel.npz`, `fields.npy`, `times.npy`, `ep_blocks.csv`, `diagnostics.json` 및 block/observable 자료가 저장됩니다. Gaussian M03은 `b=0` calibration이며 M04의 `b=1`과 비교할 때는 `b`를 맞춘 별도 run이 필요합니다.

In [ ]:
quick_runs = {}
quick_started = perf_counter()
for model_id in tqdm(MODEL_IDS, desc='Quick physics models', unit='model',
                     disable=not SHOW_PROGRESS, bar_format=BAR_FORMAT):
    config = pilot_config(model_id, n=QUICK_N, **MODEL_OVERRIDES[model_id])
    result = run_experiment(config, QUICK_RUN, output_dir=OUTPUT / 'smoke' / model_id, nn_metadata=NN_METADATA,
                            show_progress=SHOW_PROGRESS, progress_desc=f'{model_id} smoke')
    quick_runs[model_id] = result
    print(model_id, result.fields.shape, '| field mean drift:',
          float(np.max(np.abs(result.fields.mean(axis=(1, 2, 3)) - config.phi_bar))))
record_stage('quick_physics_all_models', quick_started)
print('Six-model execution complete. These short trajectories are not physical validation.')

In [ ]:
physics_rows = []
for model_id, result in quick_runs.items():
    for row in result.ep_blocks:
        physics_rows.append({'model_id': model_id, **row})
physics_table = pd.DataFrame(physics_rows)
display(physics_table[(physics_table.mechanism == 'total') & (physics_table.shell == 'all')])
# Preserve the runner's actual stationarity, mass, tail and solver diagnostics.
display({model_id: result.diagnostics for model_id, result in quick_runs.items()})
physics_table.to_csv(OUTPUT / 'smoke_physics_table.csv', index=False)

## 3. Kernel, 상태, stationarity 진단

kernel 그림에만 FFT shift를 사용합니다. 계산·저장 배열의 원점은 `(0,0)`입니다. 아래 변화량은 긴 run의 burn-in/block 길이를 정하기 위한 진단이며, 짧은 구간의 평탄한 곡선만으로 stationarity를 판정하지 않습니다. 물리 run은 slow observable의 `tau_int`, burn-in ≥ 20 tau, block ≥ 5 tau, 독립 block ≥ 30을 확인합니다.

In [ ]:
plot_started = perf_counter()
fig, axes = plt.subplots(3, len(quick_runs), figsize=(3 * len(quick_runs), 8))
odd_vmax = max(1e-12, max(float(np.max(np.abs(result.model.Ko))) for result in quick_runs.values()))
for col, (model_id, result) in enumerate(quick_runs.items()):
    model = result.model
    for row, values, title, cmap in (
        (0, np.fft.fftshift(model.K), f'{model_id}: full K', 'viridis'),
        (1, np.fft.fftshift(model.Ko), 'odd K', 'RdBu_r'),
        (2, result.fields[-1, 0], 'final phi (transient)', 'RdBu_r'),
    ):
        im = axes[row, col].imshow(values, origin='lower', cmap=cmap,
                                  **(dict(vmin=-odd_vmax, vmax=odd_vmax) if row == 1 else {}))
        axes[row, col].set_title(title)
        fig.colorbar(im, ax=axes[row, col], shrink=.6)
plt.tight_layout()
plt.savefig(OUTPUT / 'kernel_and_field.png', dpi=130)
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for model_id, result in quick_runs.items():
    fields = result.fields[:, 0]
    variance = np.mean(fields ** 2, axis=(1, 2))
    grad_energy = .5 * np.mean(fields * laplacian(fields), axis=(1, 2))
    spectrum = np.abs(np.fft.fft2(fields)) ** 2 / result.model.N
    low_q = .5 * (spectrum[:, 0, 1] + spectrum[:, 1, 0])
    for ax, values in zip(axes, (variance, grad_energy, low_q)):
        ax.plot(result.times, values, label=model_id)
for ax, title in zip(axes, ('mean(phi^2)', 'gradient energy / site', 'selected low-q S')):
    ax.set(xlabel='physical time', title=title)
axes[0].legend()
plt.tight_layout()
plt.savefig(OUTPUT / 'stationarity_observables.png', dpi=130)
plt.show()
record_stage('kernel_field_stationarity_plots', plot_started)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, (model_id, result) in zip(axes.flat, quick_runs.items()):
    active_mechanisms = {key.split(':')[0] for key in result.model.active_channels(result.fields[0, 0])}
    selected = [r for r in result.ep_blocks if r['method'] == 'midpoint'
                and r['mechanism'] in active_mechanisms
                and (r['shell'] == 'tail' or (str(r['shell']).startswith('r')
                     and int(r['shell'][1:]) <= ARCHITECTURE.max_distance))]
    mechanisms = sorted({r['mechanism'] for r in selected})
    labels = sorted({str(r['shell']) for r in selected},
                    key=lambda label: float('inf') if label == 'tail' else int(label[1:]))
    positions = {label: i for i, label in enumerate(labels)}
    for mechanism in mechanisms:
        rows = sorted((r for r in selected if r['mechanism'] == mechanism),
                      key=lambda row: positions[str(row['shell'])])
        # Shared physical coordinates; tail contains r>R and does not overlap displayed r<=R.
        xs = [positions[str(r['shell'])] for r in rows]
        ax.plot(xs, [r['mean'] for r in rows], marker='o', label=mechanism)
    ax.set_xticks(range(len(labels)),
                  [f'tail>{ARCHITECTURE.max_distance}' if label == 'tail' else label
                   for label in labels], rotation=45)
    ax.axhline(0, color='black', lw=.7)
    ax.set(title=model_id + ': physical midpoint channels', ylabel='extensive EP rate')
    if mechanisms:
        ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT / 'physical_shell_work.png', dpi=130)
plt.show()

if 'M03' in quick_runs and quick_runs['M03'].model.config.b == 0:
    gaussian = quick_runs['M03'].model.gaussian_reference()
    print('M03 stationary continuum-time Gaussian reference EPR:', gaussian['sigma_total'])
    print('Short-run estimates include sampling, transient and finite-dt error.')

## 4. 긴 실험: controls / independent seeds / dt / observation lag

`RUN_EXTENDED=True`로 선택한 군만 실행합니다. `SELECTED_CASE_NAMES=('all',)`은 해당 모델의 모든 control을 선택합니다. `control_configs`에는 M00 passive, source/receiver reciprocal·delta·uniform, M02 axis swap, M03 even/reflection, M04 full reciprocal·cancellation 등이 있습니다. `eta=1`만인 M04는 directional term이 남으므로 full equilibrium reference와 구분합니다.

아래 시간은 **출발 설정**입니다. 관측한 correlation time에 맞춰 늘려야 합니다. `dt/2`, `dt/4`에서도 같은 물리 burn-in/measurement/block 시간을 유지합니다. 기본 5 independent seeds를 사용하며 finite-size 비교는 `EXTENDED_N`을 32→64로 바꿔 별도로 실행합니다. grid refinement와 다릅니다. EPR 양성 판정에는 `sigma > 3 block_SE`뿐 아니라 stationarity 및 dt/lag 수렴이 필요합니다.

In [ ]:
RUN_EXTENDED = False
EXTENDED_MODEL_IDS = ('M00', 'M03', 'M01')
EXTENDED_N = 32
SELECTED_CASE_NAMES = ('primary',)  # e.g. ('primary', 'reciprocal') for M01; or ('all',)
SEEDS = (101, 102, 103, 104, 105)
DT_VALUES = (1e-3, 5e-4, 2.5e-4)
OBSERVATION_LAGS = (1, 4, 16)  # integration steps; record exact physical interval in each run
BURN_IN_TIME = 20.0
MEASUREMENT_TIME = 100.0
BLOCK_TIME = 1.0
OBSERVATION_STRIDE = 1  # must divide every selected observation lag

available_cases = {mid: control_configs(mid, n=EXTENDED_N, **MODEL_OVERRIDES[mid])
                   for mid in EXTENDED_MODEL_IDS}
display({mid: list(cases) for mid, cases in available_cases.items()})
extended_runs = {}  # run tag -> saved directory; do not retain all trajectories in RAM
extended_summary = []
frames_max = round(MEASUREMENT_TIME / min(DT_VALUES)) // OBSERVATION_STRIDE + 1
max_frame_gb = frames_max * EXTENDED_N ** 2 * 8 / 1e9
case_count = sum(len(cases) if SELECTED_CASE_NAMES == ('all',) else len(SELECTED_CASE_NAMES)
                 for cases in available_cases.values())
total_frame_gb = case_count * len(SEEDS) * sum(
    (round(MEASUREMENT_TIME / dt) // OBSERVATION_STRIDE + 1) * EXTENDED_N ** 2 * 8 / 1e9
    for dt in DT_VALUES)
print(f'Estimated float64 field storage: largest run {max_frame_gb:.2f} GB; total {total_frame_gb:.2f} GB.')
print('Peak RAM also includes working arrays/copies; tune durations and storage stride for your machine.')
if RUN_EXTENDED:
    extended_started = perf_counter()
    jobs = []
    for model_id, cases in available_cases.items():
        names = tuple(cases) if SELECTED_CASE_NAMES == ('all',) else SELECTED_CASE_NAMES
        for case_name in names:
            if case_name not in cases:
                raise ValueError(f'{model_id}: unknown case {case_name}; choose {list(cases)}')
            jobs.extend((model_id, case_name, dt, seed) for dt in DT_VALUES for seed in SEEDS)
    for model_id, case_name, dt, seed in tqdm(
            jobs, desc='Controls / dt / seeds', unit='run',
            disable=not SHOW_PROGRESS, bar_format=BAR_FORMAT):
        run_config = RunConfig(
            seed=seed, dt=dt, burn_in_steps=round(BURN_IN_TIME / dt),
            measurement_steps=round(MEASUREMENT_TIME / dt),
            observation_stride=OBSERVATION_STRIDE, observation_lags=OBSERVATION_LAGS,
            block_steps=round(BLOCK_TIME / dt), max_distance=ARCHITECTURE.max_distance,
        )
        tag = f'{model_id}_{case_name}_dt{dt:g}_seed{seed}'
        run_path = OUTPUT / 'extended' / tag
        result = run_experiment(available_cases[model_id][case_name], run_config,
                                output_dir=run_path, nn_metadata=NN_METADATA,
                                show_progress=SHOW_PROGRESS, progress_desc=tag)
        extended_runs[tag] = run_path
        extended_summary.extend({'run_id': tag, **row} for row in result.ep_blocks
                                if row['mechanism'] == 'total' and row['shell'] == 'all')
        del result
        tqdm.write('Saved: ' + tag)
    record_stage('extended_physics_controls_seeds_dt', extended_started)
    pd.DataFrame(extended_summary).to_csv(OUTPUT / 'extended_summary.csv', index=False)
    display(pd.DataFrame(extended_summary))
else:
    print('Extended physics runs skipped; set RUN_EXTENDED=True after choosing cases and durations.')

## 5. 저장한 실험 재개와 finite-lag endpoint 확인

`load_experiment`로 기존 자료를 다시 읽습니다. fine integration-step streaming EPR과 관측 pair의 endpoint midpoint work는 다른 유한 시간간격 추정량입니다. 아래 표는 차이를 직접 보여 줍니다. 동일 physical lag끼리 비교하려면 `lag_steps * dt`도 맞추세요.

In [ ]:
SAVED_RUN = None  # e.g. ROOT / 'results/amb_kernel/.../smoke/M01'
if SAVED_RUN is not None:
    restored = load_experiment(SAVED_RUN)
    print('Restored:', restored.model.config.model_id, restored.fields.shape)

lag_started = perf_counter()
lag_rows = []
for model_id, result in tqdm(list(quick_runs.items()), desc='Finite-lag diagnostics',
                             unit='model', disable=not SHOW_PROGRESS, bar_format=BAR_FORMAT):
    x = result.fields[:, 0]
    frame_dt = float(np.diff(result.times).mean())
    for lag in (1, 4, 16):
        work = []
        for i in range(0, len(x) - lag, lag):
            mid = .5 * (x[i] + x[i + lag])
            channels = result.model.active_channels(mid)
            active = sum(channels.values(), np.zeros_like(mid))
            work.append(-np.sum(active * (x[i + lag] - x[i])) / result.model.config.D)
        lag_rows.append({'model_id': model_id, 'lag_frames': lag,
                         'physical_lag': lag * frame_dt,
                         'endpoint_rate': np.mean(work) / (lag * frame_dt), 'n_intervals': len(work)})
lag_table = pd.DataFrame(lag_rows)
display(lag_table)
lag_table.to_csv(OUTPUT / 'finite_lag_endpoint_work.csv', index=False)
record_stage('finite_lag_diagnostics', lag_started)

## 6. 실제 학습용 independent ensemble

`DO_TRAIN=True`로 dataset 생성 → 학습 → checkpoint 복원 → test 평가를 실행합니다. **위의 짧은 `quick_runs`는 학습 데이터로 사용하지 않습니다.** 기본 데이터 규모는 현재 `Corr_labpm.ipynb`와 같은 train 1,024 / validation 256 / test 10개의 독립 trajectory, 각 1,000 interval(1,001 frame)입니다. 기본 grid는 명세의 `32×32`, `dt=0.01`, burn-in은 10,000 step입니다. 따라서 burn-in 물리 시간은 100, 측정 물리 시간은 10입니다. burn-in 길이의 충분성은 tau/stationarity로 확인해야 합니다.

`DATASET_SMOKE=True`는 연결 검증용 작은 ensemble/짧은 학습을 명시적으로 선택합니다. 실제 실험 기본값은 `False`이며 train/validation/test는 독립 seed 집합으로 생성합니다. `DATASET_ROOT`에 기존 dataset 상위 폴더를 지정하면 물리 parameter/kernel/seed/길이가 정확히 일치하는 완료 trajectory를 재사용합니다.

Dataset은 trajectory마다 저장하고 read-only memory map으로 읽습니다. 한 번에 한 모델을 생성·학습·평가한 뒤 map을 닫으며, 그림에 필요한 test pair만 메모리에 남깁니다. 기본 lag=1 pair 수는 train **1,024,000**, validation **256,000**, test **10,000**입니다. 모델당 float64 field 저장 공간은 약 **9.85 GiB**이며 전체를 RAM에 복사하지 않습니다.

`SIMULATION_DEVICE='auto'`는 CUDA가 있으면 GPU에서 여러 trajectory를 batch로 동시에 적분합니다. `SIMULATION_BATCH_SIZE=32`로 병렬 trajectory 수를 조절합니다. `device='cuda'` 또는 `'cuda:0'`을 명시했는데 CUDA를 사용할 수 없으면 오류를 내며 CPU로 몰래 전환하지 않습니다. `auto`의 선택 결과와 실제 batch 상한을 아래에서 출력합니다. CPU에서는 기존 SciPy simulator가 한 trajectory씩 적분합니다.

Simulation batch는 trajectory 병렬화 크기이고, 아래 NN의 logical/micro batch와는 별도 설정입니다. 시뮬레이션과 NN 학습은 독립적으로 CPU/GPU를 선택할 수 있습니다. GPU 메모리가 부족하면 `SIMULATION_BATCH_SIZE`를 줄이세요. 출력하는 한 개 field tensor의 크기는 메모리 하한일 뿐 solver/kernel/noise/진단 buffer를 포함한 peak VRAM 추정치가 아닙니다.

학습 기본값은 기존 LABPM 규모의 2,000 optimizer iterations, train/eval batch 2,048입니다. `micro_batch_size=32`로 나누어 gradient를 누적하여 큰 logical batch의 loss를 유지하면서 activation 메모리를 제한합니다. CUDA가 있으면 기본 사용하며 `device='cpu'`로 바꿀 수 있습니다. 시뮬레이션, dataset, train/validation/test progress에 elapsed와 ETA가 표시됩니다.

정규화는 train-only 고정 `z=(phi-m)/s`, force 복원은 `d_phi=d_z/s`입니다. 공동 extensive score **S=N×sum(branch)** 에 기존 alpha-NEEP/AdamW를 적용합니다. `alpha=-0.5` sample loss는 `4*(exp(-S/2)-1)`입니다. architecture H64/layers3/R3/ELU는 모든 모델에서 동일합니다.

In [ ]:
DO_TRAIN = False
DATASET_SMOKE = False
MODELS_TO_TRAIN = MODEL_IDS  # e.g. ('M03',) for the first Gaussian calibration
TRAIN_M, VAL_M, TEST_M = 1024, 256, 10
N_INTERVALS = 1000  # 1001 frames/trajectory, 1000 adjacent pairs at lag=1
BURN_IN_STEPS = 10000
TRAIN_N = 32
TRAIN_DT = 0.01
TRAIN_OBSERVATION_STRIDE = 1
SIMULATION_DEVICE = 'auto'  # 'auto', 'cpu', 'cuda', or e.g. 'cuda:0'
SIMULATION_BATCH_SIZE = 32  # trajectories integrated together on CUDA; CPU uses one
SPLIT_SEEDS = (12031, 23041, 34057)
DATASET_ROOT = None  # existing datasets/ parent folder to reuse exact matching replicas

TRAINING = TrainingConfig(iterations=2000, batch_size=2048, eval_batch_size=2048,
                          micro_batch_size=32, learning_rate=1e-3,
                          alpha=-0.5, record_every=100, seed=3,
                          device='cuda' if torch.cuda.is_available() else 'cpu',
                          dtype='float64', lag_frames=1)
if DATASET_SMOKE:
    DATASET = DatasetConfig.smoke()
    DATASET_N = 12
    TRAINING = replace(TRAINING, iterations=4, batch_size=8, eval_batch_size=8,
                       micro_batch_size=4, record_every=1, device='cpu')
else:
    DATASET = DatasetConfig(train_count=TRAIN_M, validation_count=VAL_M, test_count=TEST_M,
                           intervals=N_INTERVALS, burn_in_steps=BURN_IN_STEPS,
                           observation_stride=TRAIN_OBSERVATION_STRIDE, split_seeds=SPLIT_SEEDS)
    DATASET_N = TRAIN_N

DATA_RUN = RunConfig(dt=TRAIN_DT, burn_in_steps=DATASET.burn_in_steps,
                     measurement_steps=DATASET.intervals * DATASET.observation_stride,
                     observation_stride=DATASET.observation_stride,
                     observation_lags=tuple(DATASET.observation_stride * k for k in (1, 4, 16)),
                     block_steps=min(256, DATASET.intervals * DATASET.observation_stride),
                     max_distance=ARCHITECTURE.max_distance)
frame_dt = DATA_RUN.dt * DATASET.observation_stride
counts = {'train': DATASET.train_count, 'validation': DATASET.validation_count, 'test': DATASET.test_count}
resolved_simulation_device = resolve_simulation_device(SIMULATION_DEVICE)
if isinstance(SIMULATION_BATCH_SIZE, bool) or not isinstance(SIMULATION_BATCH_SIZE, int) or SIMULATION_BATCH_SIZE < 1:
    raise ValueError('SIMULATION_BATCH_SIZE must be a positive integer.')
simulation_uses_cuda = resolved_simulation_device.startswith('cuda')
effective_simulation_batch = min(SIMULATION_BATCH_SIZE, sum(counts.values())) if simulation_uses_cuda else 1
size_rows = [{'split': name, 'trajectories': count, 'frames_per_trajectory': DATASET.intervals + 1,
              'available_pairs_at_selected_lag': count * (DATASET.intervals + 1 - TRAINING.lag_frames),
              'float64_fields_GiB': count * (DATASET.intervals + 1) * DATASET_N ** 2 * 8 / 2**30}
             for name, count in counts.items()]
display(pd.DataFrame(size_rows))
per_model_gib = sum(row['float64_fields_GiB'] for row in size_rows)
print('Selected models:', ', '.join(MODELS_TO_TRAIN))
print(f'Grid {DATASET_N}x{DATASET_N}; observation interval {frame_dt:g}; NN training device {TRAINING.device}.')
print(f'Simulation device: {SIMULATION_DEVICE} -> {resolved_simulation_device}; effective parallel trajectories: up to {effective_simulation_batch}.')
print('Simulation backend:', 'batched PyTorch CUDA' if simulation_uses_cuda else 'serial SciPy CPU reference')
one_batch_field_mib = effective_simulation_batch * DATASET_N ** 2 * 8 / 2**20
print(f'One float64 phi batch tensor: {one_batch_field_mib:.3f} MiB (lower bound, not peak working memory).')
if simulation_uses_cuda:
    gpu = torch.cuda.get_device_properties(torch.device(resolved_simulation_device))
    print(f'CUDA device: {gpu.name}; total VRAM: {gpu.total_memory / 2**30:.2f} GiB.')
print(f'Field storage: {per_model_gib:.2f} GiB/model; {len(MODELS_TO_TRAIN) * per_model_gib:.2f} GiB selected total, plus diagnostics/checkpoints.')
print(f'Integration steps/model including burn-in: {sum(counts.values()) * (DATASET.burn_in_steps + DATASET.intervals * DATASET.observation_stride):,}.')
print('Preset:', 'explicit small dataset/learning smoke' if DATASET_SMOKE else 'full independent LABPM-sized training ensemble')
print(f'Logical train/eval batches: {TRAINING.batch_size}/{TRAINING.eval_batch_size}; micro batch: {TRAINING.micro_batch_size}; iterations: {TRAINING.iterations}.')

dataset_root = Path(DATASET_ROOT) if DATASET_ROOT is not None else OUTPUT / 'datasets'
trained, evaluations, test_examples, training_models, training_frame_dt = {}, {}, {}, {}, {}
training_sources = {}
(OUTPUT / 'training_plan.json').write_text(json.dumps({
    'enabled': DO_TRAIN, 'dataset_smoke': DATASET_SMOKE, 'model_ids': MODELS_TO_TRAIN,
    'dataset_config': asdict(DATASET), 'grid_n': DATASET_N, 'run_config': asdict(DATA_RUN),
    'simulation_device_requested': SIMULATION_DEVICE,
    'simulation_device_resolved': resolved_simulation_device,
    'simulation_batch_size': SIMULATION_BATCH_SIZE,
    'effective_simulation_batch_upper_bound': effective_simulation_batch,
    'training_config': asdict(TRAINING), 'architecture': vars(ARCHITECTURE.options()),
    'dataset_root': str(dataset_root), 'size_estimates': size_rows,
}, indent=2), encoding='utf8')

if DO_TRAIN:
    pipeline_started = perf_counter()
    for model_id in tqdm(MODELS_TO_TRAIN, desc='Dataset / train / test models', unit='model',
                         disable=not SHOW_PROGRESS, bar_format=BAR_FORMAT):
        physical = AMBKernelModel(pilot_config(model_id, n=DATASET_N, **MODEL_OVERRIDES[model_id]))
        generation_started = perf_counter()
        dataset = generate_dataset(physical, dataset_root / model_id,
                                   dataset_config=DATASET, run_config=DATA_RUN,
                                   show_progress=SHOW_PROGRESS, resume=True, nn_metadata=NN_METADATA,
                                   device=SIMULATION_DEVICE, simulation_batch_size=SIMULATION_BATCH_SIZE)
        record_stage(f'{model_id}/independent_dataset', generation_started)
        try:
            splits = dataset.training_splits()
            training_sources[model_id] = str(dataset.output_dir)
            train_started = perf_counter()
            fit = train_shell_force(splits, OUTPUT / 'learning' / model_id,
                                    architecture=ARCHITECTURE, config=TRAINING,
                                    physical_metadata=dataset.manifest,
                                    show_progress=SHOW_PROGRESS, progress_desc=f'{model_id} training')
            record_stage(f'{model_id}/training_with_validation', train_started)
            # Reload the best persisted network; keep complete training history for plots.
            fit.model = load_checkpoint(fit.checkpoint, device=TRAINING.device).model
            trained[model_id] = fit
            evaluation_started = perf_counter()
            evaluations[model_id] = evaluate_shell_force(
                fit, splits.test, dataset.model, frame_dt,
                lag_frames=TRAINING.lag_frames, batch_size=TRAINING.eval_batch_size,
                micro_batch_size=TRAINING.micro_batch_size, block_pairs=32,
                output_dir=OUTPUT / 'learning' / model_id,
                show_progress=SHOW_PROGRESS, progress_desc=f'{model_id} test evaluation')
            record_stage(f'{model_id}/test_evaluation', evaluation_started)
            # Only the selected test pair survives; close every large dataset mmap below.
            test_examples[model_id] = np.array(splits.test[0][:TRAINING.lag_frames + 1], copy=True)
            training_models[model_id] = dataset.model
            training_frame_dt[model_id] = frame_dt
            tqdm.write(f'{model_id} best checkpoint: {fit.checkpoint}')
            display(pd.DataFrame(evaluations[model_id]['rows']))
            del splits
        finally:
            dataset.close()
            del dataset
    record_stage('all_selected_training_models', pipeline_started)
else:
    print('Training disabled. Set DO_TRAIN=True to generate the independent ensemble and train; quick_runs is never used for training.')

## 7. NN total / raw branch / known-force projection / curl

- `NN_total`: N배 공간 평균 score를 observation interval로 나눈 extensive rate.
- `endpoint_midpoint`: 동일 test pair에서 알려진 active work. streaming truth와 별도입니다.
- `NN_raw_branch`: network가 선택한 additive decomposition; 물리 채널과 같다고 가정하지 않습니다.
- `NN_force_projection`: **−〈nu_cᵀ L d_theta,total〉**, 왼쪽 endpoint의 equal-time 진단. 이미 rate이므로 lag로 다시 나누지 않습니다.
- projected curl은 zero-mean 방향의 bilinear test로 `J_d−J_dᵀ = −(J_mu−J_muᵀ)/D`를 검사합니다. total curl만으로 branch attribution이나 global exactness를 증명하지 않습니다.

block SE는 인접 pair를 독립 표본으로 세는 naive SEM이 아닙니다. 선택한 block 길이가 tau에 비해 충분한지 따로 확인해야 합니다. M05의 `u*u_r`는 attribution convention이며 `r=0` physical channel은 다른 shell 정보에도 의존하므로 NN on-site null과 다릅니다.

In [ ]:
if trained:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    analysis_started = perf_counter()
    for model_id, fit in tqdm(list(trained.items()), desc='Learned-force curl checks', unit='model',
                              disable=not SHOW_PROGRESS, bar_format=BAR_FORMAT):
        history = pd.DataFrame(fit.history)
        axes[0].plot(history.iteration, history.validation_loss, label=model_id)
        rows = evaluations[model_id]['rows']
        raw = [r for r in rows if r['method'] == 'NN_raw_branch']
        axes[1].plot(range(len(raw)), [r['mean'] for r in raw], marker='o', label=model_id)
        phi = test_examples[model_id][0, 0]
        curl = projected_curl_check(fit, training_models[model_id], phi, probes=8,
                                    show_progress=SHOW_PROGRESS)
        display({'model_id': model_id, 'projected_total_curl': curl})
        (OUTPUT / 'learning' / model_id / 'projected_curl.json').write_text(
            json.dumps(curl, indent=2), encoding='utf8')
    axes[0].set(xlabel='iteration', ylabel='joint extensive validation loss')
    axes[1].set(xlabel='Chebyshev shell r', ylabel='signed NN raw extensive EP rate')
    axes[1].axhline(0, color='black', lw=.7)
    for ax in axes:
        ax.legend()
    plt.tight_layout()
    plt.savefig(OUTPUT / 'learning_curves_and_shells.png', dpi=130)
    plt.show()
    record_stage('learned_force_curl_and_curves', analysis_started)

In [ ]:
if trained:
    model_id = next(iter(trained))
    fit, physical = trained[model_id], training_models[model_id]
    test = test_examples[model_id]
    lag = TRAINING.lag_frames
    pair_phi = np.stack((test[0, 0], test[lag, 0]))
    parameter = next(fit.model.parameters())
    z = fit.normalizer.transform(torch.as_tensor(pair_phi, dtype=torch.float64)[None])
    with torch.no_grad():
        maps = fit.model(z.to(parameter.device, parameter.dtype), return_maps=True)[0].cpu().numpy()
    delta_t = lag * training_frame_dt[model_id]
    pred_local_rate = maps.sum(axis=0) / delta_t
    midpoint, delta = pair_phi.mean(axis=0), pair_phi[1] - pair_phi[0]
    channels = physical.active_channels(midpoint)
    known_active = sum(channels.values(), np.zeros_like(midpoint))
    truth_local_rate = -known_active * delta / (physical.config.D * delta_t)
    vmax = max(np.max(np.abs(pred_local_rate)), np.max(np.abs(truth_local_rate)), 1e-12)
    fig, axes = plt.subplots(1, 3, figsize=(13, 4))
    for ax, values, title in zip(axes, (midpoint, truth_local_rate, pred_local_rate),
                                 ('phi midpoint', 'known endpoint work / site / time', 'NN local rate')):
        image = ax.imshow(values, origin='lower', cmap='RdBu_r',
                          **({} if title == 'phi midpoint' else dict(vmin=-vmax, vmax=vmax)))
        ax.set_title(model_id + ': ' + title)
        fig.colorbar(image, ax=ax, shrink=.7)
    plt.tight_layout()
    plt.savefig(OUTPUT / f'{model_id}_local_ep_comparison.png', dpi=130)
    plt.show()
    print('Spatial SUM of each rate map gives its extensive total; no additional N factor.')

In [ ]:
# Per-stage timings exclude waits between notebook cells; total is wall time since imports.
timing_table = pd.DataFrame(STAGE_TIMINGS)
display(timing_table)
timing_table.to_csv(OUTPUT / 'stage_timings.csv', index=False)
total_elapsed = perf_counter() - NOTEBOOK_STARTED
print(f'Notebook total elapsed: {timedelta(seconds=round(total_elapsed))} ({total_elapsed:.1f} s).')
(OUTPUT / 'elapsed_summary.json').write_text(json.dumps({
    'notebook_wall_seconds': total_elapsed,
    'stages': STAGE_TIMINGS,
    'note': 'Total includes idle time between cells; nested pipeline totals are not additive.',
}, indent=2), encoding='utf8')

## 결과를 해석하기 전

1. 원본 명세의 algebra checks와 simulator의 noise/mass/kernel/trace tests를 통과했는지 확인합니다.
2. passive/reciprocal/null controls, stationarity, autocorrelation 기반 burn-in·block, independent seeds, dt/lag 수렴을 검토합니다. Gaussian M03에서는 analytic covariance/EPR도 비교합니다.
3. full kernel tail, 모델별 표현 한계, 총량과 밀도, 유한 관측 간격을 구분한 뒤 NN 결과를 비교합니다. H/R을 모델별로 조절해 결과를 맞추지 않습니다.
4. 총 EPR 일치와 raw branch attribution 일치는 별개입니다. source quadratic-rank 및 receiver cross-shell 의존성을 남은 오차의 가능한 원인으로 기록합니다.

모든 그림과 표는 이 notebook 실행의 `OUTPUT` 아래에 저장됩니다. checkpoint는 `learning/<model>/best_model_parameter.pth.tar`, 생성자·정규화·loss·split·source hash는 `learning_manifest.json`에서 확인할 수 있습니다.